In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 1 — DEPENDANCES
# Objectif: installer uniquement le runtime necessaire pour produire les JSON.
# Entrees: environnement Python.
# Sorties: librairies installees.
# Regles: pas openpyxl, pas pandas, pas export Excel.
# ════════════════════════════════════════════════════════════
%pip install -q -U 'transformers>=4.57.0' accelerate pymupdf pillow psutil
print('✅ Dependances OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 2 — IMPORTS
# Objectif: charger les librairies nécessaires au pipeline JSON.
# Entrées: librairies installées.
# Sorties: namespaces importés.
# Règles: suppression des imports Excel/contrôles.
# ════════════════════════════════════════════════════════════
import time, json, re, gc, copy, unicodedata
import numpy as np
from pathlib import Path
from datetime import datetime
import fitz, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
print('✅ Imports OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 3 — CONFIG
# Objectif: chemins, paramètres modèle, batch, dossiers de sortie JSON.
# Entrées: dossier PDF.
# Sorties: liste pdfs, chemins JSON_DIR/LOG/INDEX.
# Règles: JSON uniquement, pas de sortie Excel.
# ════════════════════════════════════════════════════════════
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_NEW_TOKENS = 7000
IMAGE_MAX_SIZE = 2024
MIN_PIXELS = 4 * 32 * 32
MAX_PIXELS = 2000 * 32 * 32
PDF_ZOOM = 3.0
BLANK_THRESHOLD = 0.95
CLASSIF_BATCH_SIZE = 16
GPU_BATCH_SIZE = 2
INPUT_DIR = Path('/mnt/data/bilans_in')
OUTPUT_DIR = Path('/mnt/data/bilans_out')
JSON_DIR = OUTPUT_DIR / 'json_bilans_v8'
LOG_PATH = OUTPUT_DIR / 'pipeline_bilans_v8.log'
INDEX_PATH = OUTPUT_DIR / 'index_bilans_v8.jsonl'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print(f'Device: {DEVICE} | PDF: {len(pdfs)} | JSON: {JSON_DIR}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 4 — LOG
# Objectif: journaliser le pipeline dans un fichier et stdout.
# Entrées: message.
# Sorties: ligne horodatée.
# Règles: utile pour audit et reprise.
# ════════════════════════════════════════════════════════════
def log(msg):
    ligne = datetime.now().strftime('%Y-%m-%d %H:%M:%S') + ' — ' + str(msg)
    print(ligne)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(ligne + chr(10))
print('✅ Log OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 5 — CHARGEMENT MODELE
# Objectif: charger Qwen3.6-VL FP8 dequantisé en bf16.
# Entrées: MODEL_PATH.
# Sorties: processor, model.
# Règles: padding gauche, génération déterministe.
# ════════════════════════════════════════════════════════════
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = 'left'
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, dtype=torch.bfloat16, device_map='auto', trust_remote_code=True, low_cpu_mem_usage=True, quantization_config=FP8Config(dequantize=True))
model.eval()
print(f'✅ Modele charge en {time.time()-t0:.1f}s')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 6 — UTILITAIRES
# Objectif: rendu PDF, deskew, détection pages blanches, inférence VLM.
# Entrées: PDF, images.
# Sorties: pages images, réponses VLM.
# Règles: pages blanches détectées localement, pas envoyées au VLM.
# ════════════════════════════════════════════════════════════
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)

def strip_accents(s):
    s = str(s)
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))

def norm_key(s):
    s = strip_accents(str(s)).lower()
    s = re.sub('[^a-z0-9]+', '_', s)
    return s.strip('_')

def estimate_skew(img):
    small = img.convert('L').copy()
    small.thumbnail((500, 500))
    def score(a):
        r = np.array(small.rotate(a, expand=True, fillcolor=255)) < 128
        proj = r.sum(axis=1)
        return float((proj ** 2).sum())
    best = max(range(-12, 13, 2), key=score)
    best = max([best-1, best-0.5, best, best+0.5, best+1], key=score)
    return best if abs(best) >= 1 else 0.0

def deskew(img):
    a = estimate_skew(img)
    if a:
        img = img.rotate(a, expand=True, fillcolor=(255,255,255), resample=Image.BICUBIC)
    return img, a

def is_blank(image, threshold=BLANK_THRESHOLD):
    arr = np.array(image.convert('L'))
    return (arr > 240).sum() / arr.size >= threshold

def pdf_to_pages(path, zoom=PDF_ZOOM):
    doc = fitz.open(path)
    matrix = fitz.Matrix(zoom, zoom)
    pages = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
        img, angle = deskew(img)
        img = resize(img)
        pages.append({'index': i, 'image': img, 'largeur_px': img.width, 'hauteur_px': img.height, 'rotation_estimee': angle})
    doc.close()
    return pages

def parse_json(text):
    if not text:
        return {}
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find('{')
    end = text.rfind('}')
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            return {}
    return {}

def apply_template(messages):
    try:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def _decode(out_i, in_len):
    return processor.decode(out_i[in_len:], skip_special_tokens=True, clean_up_tokenization_spaces=False)

def ask_single(prompt, image):
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    inputs = processor(text=[apply_template(msgs)], images=[image], return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    return {'text': _decode(out[0], inputs['input_ids'].shape[1]), 'tokens_in': int(inputs['input_ids'].shape[1]), 'tokens_out': int(out[0].shape[0] - inputs['input_ids'].shape[1]), 'elapsed': round(time.time()-t0, 2)}

def ask_batch(prompt, images):
    if not images:
        return []
    if len(images) == 1:
        return [ask_single(prompt, images[0])]
    msgs = [[{'role':'user','content':[{'type':'image','image':img},{'type':'text','text':prompt}]}] for img in images]
    inputs = processor(text=[apply_template(m) for m in msgs], images=images, return_tensors='pt', padding=True).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    el = time.time() - t0
    in_len = inputs['input_ids'].shape[1]
    attn = inputs.get('attention_mask')
    return [{'text': _decode(out[i], in_len), 'tokens_in': int(attn[i].sum().item()) if attn is not None else in_len, 'tokens_out': int(out[i].shape[0] - in_len), 'elapsed': round(el/len(images), 2)} for i in range(len(images))]

print('✅ Utilitaires OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 7 — SCHEMAS + PROMPTS
# Objectif: définir le contrat de données et les prompts VLM.
# Entrées: SCHEMAS canoniques.
# Sorties: prompts classification/extraction.
# Règles: cachet proche montant => incertain=true.
# ════════════════════════════════════════════════════════════
SCHEMAS = {
    'ACTIF': {
        'cols': ['montant_brut', 'amortissements_provisions_pertes', 'net_n', 'net_n1'],
        'postes': {
            'ecarts_acquisition_goodwill': 'Ecart d acquisition goodwill',
            'immobilisations_incorporelles': 'Immobilisations incorporelles',
            'terrains': 'Terrains',
            'batiments': 'Batiments',
            'autres_immobilisations_corporelles': 'Autres Immobilisations corporelles',
            'immobilisations_en_concession': 'Immobilisations en concession',
            'immobilisations_en_cours': 'Immobilisations en cours',
            'titres_mis_en_equivalence': 'Titres mis en equivalence',
            'autres_participations_creances': 'Autres participations et creances rattachees',
            'autres_titres_immobilises': 'Autres titres immobilises',
            'prets_actifs_financiers_non_courants': 'Prets et autres actifs financiers non courants',
            'impots_differes_actif': 'Impots Differes Actif',
            'total_actif_non_courant': 'TOTAL ACTIF NON COURANT',
            'stocks_encours': 'Stocks et encours',
            'clients': 'Clients',
            'autres_debiteurs': 'Autres debiteurs',
            'impots_assimiles_actif': 'Impots & Assimiles',
            'autres_creances_assimiles': 'Autres Creances & Emplois assimiles',
            'placements_financiers_courants': 'Placements et autres actifs financiers courants',
            'tresorerie_actif': 'Tresorerie',
            'total_actif_courant': 'TOTAL ACTIF COURANT',
            'total_general_actif': 'TOTAL GENERAL ACTIF'
        }
    },
    'PASSIF': {
        'cols': ['n', 'n1'],
        'postes': {
            'capital_emis': 'Capital emis',
            'capital_non_appele': 'Capital non appele',
            'primes_reserves': 'Primes et reserves',
            'ecart_reevaluation': 'Ecart de reevaluation',
            'ecart_equivalence': 'Ecart d equivalence',
            'resultat_net_passif': 'Resultat net',
            'report_a_nouveau': 'Report a nouveau',
            'part_societe_consolidante': 'Part de la societe consolidante',
            'part_minoritaires': 'Part des minoritaires',
            'total_capitaux_propres': 'TOTAL I',
            'emprunts_dettes_financieres': 'Emprunts et dettes financieres',
            'impots_differes_provisionnes': 'Impots differes et provisionnes',
            'autres_dettes_non_courantes': 'Autres dettes non courantes',
            'provisions_produits_avance': 'Provisions et produits comptabilises d avance',
            'total_passifs_non_courants': 'TOTAL PASSIFS NON COURANTS II',
            'fournisseurs_rattaches': 'Fournisseurs et comptes rattaches',
            'impots_passif': 'Impots',
            'autres_dettes': 'Autres dettes',
            'tresorerie_passif': 'Tresorerie Passif',
            'total_passifs_courants': 'TOTAL PASSIFS COURANTS',
            'total_general_passif': 'TOTAL GENERAL PASSIF'
        }
    },
    'TCR': {
        'cols': ['n_debit', 'n_credit', 'n1_debit', 'n1_credit'],
        'postes': {
            'ventes_marchandises': 'Ventes de Marchandises',
            'produits_fabriques': 'Produits Fabriques',
            'prestations_services': 'Prestations de Services',
            'ventes_travaux': 'Ventes de Travaux',
            'produits_annexes': 'Produits Annexes',
            'rabais_remises_ristournes_accordes': 'Rabais remises ristournes accordes',
            'chiffre_affaires_net': 'Chiffre d affaires net',
            'production_stockee_destockee': 'Production Stockee ou destockee',
            'production_immobilisee': 'Production immobilisee',
            'subvention_exploitation': 'Subvention d exploitation',
            'production_exercice': 'I-Production de l exercice',
            'achats_marchandises_vendues': 'Achats de Marchandises vendues',
            'matieres_premieres': 'Matieres premieres',
            'autres_approvisionnements': 'Autres Approvisionnements',
            'variation_stocks': 'Variation des Stocks',
            'achats_etudes_prestations': 'Achats d Etudes et de Prestations de services',
            'autres_consommations': 'Autres consommations',
            'sous_traitance_generale': 'Sous-traitance generale',
            'locations': 'Locations',
            'entretien_reparations': 'Entretien reparations et maintenance',
            'primes_assurances': 'Primes d assurances',
            'personnel_exterieur': 'Personnel exterieur a l entreprise',
            'remuneration_intermediaires': 'Remuneration d intermediaires et honoraires',
            'publicite': 'Publicite',
            'deplacements_missions': 'Deplacements missions et receptions',
            'autres_services': 'Autres services',
            'consommations_exercice': 'II-Consommations de l exercice',
            'valeur_ajoutee_exploitation': 'III-Valeur ajoutee d exploitation',
            'charges_personnel': 'Charges de personnel',
            'impots_taxes_assimiles': 'Impots et taxes et versements assimiles',
            'excedent_brut_exploitation': 'IV-Excedent brut d exploitation',
            'autres_produits_operationnels': 'Autres produits operationnels',
            'autres_charges_operationnelles': 'Autres charges operationnelles',
            'dotations_amortissements': 'Dotations aux amortissements',
            'provisions': 'Provisions',
            'pertes_valeur': 'Perte de Valeur',
            'reprises_pertes_valeur_provisions': 'Reprise sur pertes de valeur et provisions',
            'resultat_operationnel': 'V-Resultat operationnel',
            'produits_financiers': 'Produits financiers',
            'charges_financieres': 'Charges financieres',
            'resultat_financier': 'VI-Resultat Financier',
            'resultat_ordinaire': 'VII-Resultat ordinaire',
            'elements_extraordinaires_produits': 'Elements extraordinaires Produits',
            'elements_extraordinaires_charges': 'Elements extraordinaires Charges',
            'resultat_extraordinaire': 'VIII-Resultat extraordinaire',
            'impots_exigibles_resultats': 'Impots exigibles sur resultats',
            'impots_differes_resultats': 'Impots differes sur resultats',
            'resultat_net_exercice': 'RESULTAT NET DE L EXERCICE'
        }
    }
}

RULES = [
    'REGLES: JSON valide uniquement, sans markdown, sans backticks.',
    'Aucune valeur inventee. Absent=null. Illisible=null.',
    'IMPORTANT: si un montant est a cote, sous, recouvert ou proche d un cachet/tampon/signature/annotation, meme s il est lisible, retourner incertain=true pour ce montant.',
    'Pour chaque montant, retourner un objet avec les champs valeur, valeur_brute, incertain, commentaire.',
    'valeur est un nombre JSON sans separateur de milliers. valeur_brute est le texte lu ou null.'
]

PROMPT_CLASSIF = 'Page d un dossier fiscal algerien Serie G. Reponds UN seul mot: ACTIF, PASSIF, TCR, DECL, ANNEXE, AUTRE.'

def build_core_prompt(table, titre, colonnes):
    L = []
    L.append('Page ' + titre + ' d une liasse fiscale algerienne Serie G.')
    L.append('Extrais le tableau en JSON strict.')
    L.append('Structure: type_page, numero_page_imprimee, titre_page, entete, elements_visuels, table.')
    L.append('entete contient entreprise, nif, exercice, exercice_du, exercice_au, adresse, activite, serie_g.')
    L.append('elements_visuels contient cachet_present, tampon_present, signature_presente, annotation_manuscrite, cachet_proche_montants.')
    L.append('table contient table_code et lignes. Chaque ligne contient row_code, libelle_imprime, valeurs.')
    L.append('valeurs contient les colonnes ' + ', '.join(colonnes) + '. Chaque valeur est un objet valeur/valeur_brute/incertain/commentaire.')
    L.append('Row_code autorises:')
    for k, label in SCHEMAS[table]['postes'].items():
        L.append('- ' + k + ' : ' + label)
    L.append('Retourne seulement les row_code avec au moins une valeur non nulle.')
    L.extend(RULES)
    return chr(10).join(L)

PROMPT_ACTIF = build_core_prompt('ACTIF', 'BILAN ACTIF', ['montant_brut', 'amortissements_provisions_pertes', 'net_n', 'net_n1'])
PROMPT_PASSIF = build_core_prompt('PASSIF', 'BILAN PASSIF', ['n', 'n1'])
PROMPT_TCR = build_core_prompt('TCR', 'COMPTE DE RESULTAT', ['n_debit', 'n_credit', 'n1_debit', 'n1_credit'])

L = []
L.append('Page DECLARATION IBS / TAXE LOCALE d une liasse fiscale algerienne Serie G.')
L.append('Extrais toutes les informations lisibles en JSON strict.')
L.append('Structure: type_page, numero_page_imprimee, titre_page, entete, elements_visuels, decl.')
L.append('decl doit contenir annee_souscription, periode_imposition_du, periode_imposition_au, exercice_du, exercice_au, resultat_exercice, identification, tenue_comptabilite, certification, recap_imposition, tap, taxe_locale, autres_infos.')
L.append('Les montants doivent etre des objets valeur/valeur_brute/incertain/commentaire.')
L.extend(RULES)
PROMPT_DECL = chr(10).join(L)

L = []
L.append('Page ANNEXE d une liasse fiscale algerienne Serie G.')
L.append('Extrais tous les tableaux lisibles en JSON strict.')
L.append('Structure: type_page, sous_type_page, annexe_numero, titre_page, numero_page_imprimee, entete, elements_visuels, tables, texte_libre.')
L.append('tables est une liste. Chaque table contient table_libelle, colonnes, lignes.')
L.append('Chaque ligne contient libelle_imprime et valeurs. valeurs est un objet par colonne avec valeur/valeur_brute/incertain/commentaire.')
L.append('N oublie pas les totaux et sous-totaux.')
L.extend(RULES)
PROMPT_ANNEXE = chr(10).join(L)

L = []
L.append('Page AUTRE d un dossier fiscal algerien.')
L.append('Extrais toutes les informations lisibles en JSON strict.')
L.append('Structure: type_page, sous_type_page, titre_page, numero_page_imprimee, entete, elements_visuels, tables, texte_libre.')
L.append('Si un tableau est present, extrais-le dans tables.')
L.append('Pour les montants, utilise des objets valeur/valeur_brute/incertain/commentaire.')
L.extend(RULES)
PROMPT_AUTRE = chr(10).join(L)

EXTRACTION_PROMPTS = {
    'ACTIF': PROMPT_ACTIF,
    'PASSIF': PROMPT_PASSIF,
    'TCR': PROMPT_TCR,
    'DECL': PROMPT_DECL,
    'ANNEXE': PROMPT_ANNEXE,
    'AUTRE': PROMPT_AUTRE
}

VALID_TYPES = {'ACTIF', 'PASSIF', 'TCR', 'DECL', 'ANNEXE', 'AUTRE', 'BLANCHE'}
print('✅ Schemas + prompts OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 8 — NORMALISATION
# Objectif: transformer les sorties VLM en valeurs typées.
# Entrées: JSON VLM.
# Sorties: valeurs normalisées, objets valeur, flag incertain.
# Règles: aucune valeur inventée, null si absent.
# ════════════════════════════════════════════════════════════
def norm_str(v):
    if v is None:
        return None
    if isinstance(v, bool):
        return None
    s = re.sub('[\s]+', ' ', str(v).strip())
    if not s or s.lower() in ('null', 'none', 'n/a', 'na', '-', ''):
        return None
    return s

def norm_montant(v):
    if v is None:
        return None
    if isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip()
    if s.lower() in ('null', 'none', 'n/a', 'na', '-', ''):
        return None
    neg = (s.startswith('(') and s.endswith(')')) or s.startswith('-')
    s = re.sub('[^0-9.,-]', '', s)
    if not s:
        return None
    if s.count(',') == 1 and '.' not in s:
        s = s.replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '')
    elif s.count('.') > 1:
        s = s.replace('.', '')
    try:
        return -float(s) if neg else float(s)
    except Exception:
        return None

def norm_int(v):
    if v is None:
        return None
    if isinstance(v, bool):
        return None
    if isinstance(v, int):
        return v
    if isinstance(v, float):
        return int(v) if v.is_integer() else None
    s = norm_str(v)
    if not s:
        return None
    m = re.search('[0-9]+', s)
    return int(m.group()) if m else None

def norm_bool(v):
    if v is None:
        return False
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return bool(v)
    s = str(v).strip().lower()
    return s in ('true', '1', 'oui', 'yes', 'v', 'vrai')

def norm_nif(v):
    s = norm_str(v)
    return re.sub('[^0-9]', '', s) if s else None

def normalise_value(raw, page_number, colonne=None):
    valeur = None
    valeur_brute = None
    incertain = False
    commentaire = None
    if isinstance(raw, dict):
        valeur = norm_montant(raw.get('valeur', raw.get('valeur_brute')))
        valeur_brute = norm_str(raw.get('valeur_brute'))
        incertain = norm_bool(raw.get('incertain'))
        commentaire = norm_str(raw.get('commentaire'))
    elif raw is not None:
        valeur = norm_montant(raw)
        if isinstance(raw, (int, float)):
            valeur_brute = str(raw)
        else:
            valeur_brute = norm_str(raw)
    if valeur_brute and valeur is None:
        incertain = True
    return {'valeur': valeur, 'valeur_brute': valeur_brute, 'source_page': page_number, 'colonne_imprimee': colonne, 'incertain': incertain, 'commentaire': commentaire}

def normalise_entete(data):
    ent = data.get('entete') or {}
    if not isinstance(ent, dict):
        ent = {}
    return {
        'entreprise': norm_str(ent.get('entreprise')),
        'nif': norm_nif(ent.get('nif')),
        'exercice': norm_str(ent.get('exercice')),
        'exercice_du': norm_str(ent.get('exercice_du')),
        'exercice_au': norm_str(ent.get('exercice_au')),
        'adresse': norm_str(ent.get('adresse')),
        'activite': norm_str(ent.get('activite')),
        'serie_g': norm_str(ent.get('serie_g'))
    }

def normalise_elements_visuels(data):
    vis = data.get('elements_visuels') or {}
    if not isinstance(vis, dict):
        vis = {}
    return {
        'cachet_present': norm_bool(vis.get('cachet_present')),
        'tampon_present': norm_bool(vis.get('tampon_present')),
        'signature_presente': norm_bool(vis.get('signature_presente')),
        'annotation_manuscrite': norm_bool(vis.get('annotation_manuscrite')),
        'cachet_proche_montants': norm_bool(vis.get('cachet_proche_montants')),
        'tampon_proche_montants': norm_bool(vis.get('tampon_proche_montants')),
        'signature_proche_montants': norm_bool(vis.get('signature_proche_montants')),
        'annotation_proche_montants': norm_bool(vis.get('annotation_proche_montants'))
    }

def stamp_proche(visuels):
    if not isinstance(visuels, dict):
        return False
    return any(bool(v) for k, v in visuels.items() if k.endswith('proche_montants'))

def set_incertain_recursive(obj):
    if isinstance(obj, dict):
        if 'valeur' in obj:
            obj['incertain'] = True
        for v in obj.values():
            set_incertain_recursive(v)
    elif isinstance(obj, list):
        for v in obj:
            set_incertain_recursive(v)

def normalise_core_table(data, table, page_number):
    spec = SCHEMAS[table]
    out = {}
    raw_table = data.get('table') or {}
    if isinstance(raw_table, list):
        raw_table = raw_table[0] if raw_table else {}
    if not isinstance(raw_table, dict):
        raw_table = {}
    tables = data.get('tables') or []
    if not raw_table and isinstance(tables, list) and tables and isinstance(tables[0], dict):
        raw_table = tables[0]
    lignes = raw_table.get('lignes') or data.get('lignes') or []
    if not isinstance(lignes, list):
        lignes = []
    label_to_key = {norm_key(v): k for k, v in spec['postes'].items()}
    row_map = {}
    for ligne in lignes:
        if not isinstance(ligne, dict):
            continue
        row_code = norm_str(ligne.get('row_code'))
        if row_code not in spec['postes']:
            libelle = norm_str(ligne.get('libelle_imprime'))
            if libelle and label_to_key.get(norm_key(libelle)):
                row_code = label_to_key.get(norm_key(libelle))
        if row_code in spec['postes']:
            row_map[row_code] = ligne
    for key in spec['postes']:
        ligne = row_map.get(key) or {}
        valeurs = ligne.get('valeurs') or {}
        if not isinstance(valeurs, dict):
            valeurs = {}
        out[key] = {}
        for col in spec['cols']:
            raw_val = valeurs.get(col)
            out[key][col] = normalise_value(raw_val, page_number, col)
    return out

def normalise_donnees(page_type, data, page_number):
    donnees = {'brut': data}
    if page_type in ('ACTIF', 'PASSIF', 'TCR'):
        donnees[page_type.lower()] = normalise_core_table(data, page_type, page_number)
    return donnees

def count_values(obj):
    count = 0
    if isinstance(obj, dict):
        if 'valeur' in obj and 'source_page' in obj:
            return 1 if obj.get('valeur') is not None else 0
        for v in obj.values():
            count += count_values(v)
    elif isinstance(obj, list):
        for v in obj:
            count += count_values(v)
    return count

print('✅ Normalisation OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 9 — CONSTRUCTION PAGES JSON
# Objectif: construire chaque page du JSON final.
# Entrées: page PDF, type, données VLM.
# Sorties: objet page, synthèse, document JSON.
# Règles: page scannée toujours tracée, pages blanches non VLM.
# ════════════════════════════════════════════════════════════
def parse_type(text):
    t = (text or '').upper()
    for k in ('ACTIF', 'PASSIF', 'TCR', 'DECL', 'ANNEXE', 'AUTRE'):
        if k in t:
            return k
    return 'AUTRE'

def build_blank_page(page, fichier_source):
    page_number = page['index'] + 1
    return {
        'page_id': 'p' + str(page_number).zfill(3),
        'pdf_page_index': page['index'],
        'numero_page_scannee': page_number,
        'numero_page_imprimee': None,
        'fichier_source': fichier_source,
        'classification': {
            'type_page': 'BLANCHE',
            'type_classement_initial': 'BLANCHE',
            'sous_type_page': 'PAGE_VIERGE',
            'titre_page': None,
            'annexe_numero': None,
            'page_annexe': False,
            'page_utile': False,
            'page_blanche': True
        },
        'image': {
            'largeur_px': page.get('largeur_px'),
            'hauteur_px': page.get('hauteur_px'),
            'rotation_estimee': page.get('rotation_estimee'),
            'deskew_applique': bool(page.get('rotation_estimee'))
        },
        'statut_extraction': {
            'statut': 'BLANCHE',
            'exhaustivite': None,
            'nb_tableaux': 0,
            'nb_champs_extraits': 0,
            'commentaire': 'Page blanche detectee, non envoyee au VLM.'
        },
        'entete_page': {},
        'elements_visuels': {},
        'donnees': {},
        'tokens_in': 0,
        'tokens_out': 0,
        'temps_s': 0.0
    }

def build_page_object(page, assigned_type, data, rep, fichier_source):
    page_number = page['index'] + 1
    data = data if isinstance(data, dict) else {}
    model_type = norm_str(data.get('type_page'))
    if model_type:
        model_type = model_type.upper()
    final_type = model_type if model_type in VALID_TYPES else assigned_type
    entete = normalise_entete(data)
    visuels = normalise_elements_visuels(data)
    donnees = normalise_donnees(final_type, data, page_number)
    if stamp_proche(visuels):
        set_incertain_recursive(donnees)
    statut = 'OK' if data else 'ECHEC_EXTRACTION'
    nb_champs = count_values(donnees)
    nb_tableaux = 1 if final_type in ('ACTIF', 'PASSIF', 'TCR', 'DECL') else 0
    brut = donnees.get('brut')
    if final_type not in ('ACTIF', 'PASSIF', 'TCR', 'DECL') and isinstance(brut, dict):
        tables = brut.get('tables') or []
        if isinstance(tables, list):
            nb_tableaux = len(tables)
    annexe_numero = norm_int(data.get('annexe_numero'))
    sous_type = norm_str(data.get('sous_type_page'))
    if final_type == 'ANNEXE' and not sous_type and annexe_numero:
        sous_type = 'ANNEXE_' + str(annexe_numero)
    return {
        'page_id': 'p' + str(page_number).zfill(3),
        'pdf_page_index': page['index'],
        'numero_page_scannee': page_number,
        'numero_page_imprimee': norm_int(data.get('numero_page_imprimee')),
        'fichier_source': fichier_source,
        'classification': {
            'type_page': final_type,
            'type_classement_initial': assigned_type,
            'sous_type_page': sous_type,
            'titre_page': norm_str(data.get('titre_page')),
            'annexe_numero': annexe_numero,
            'page_annexe': final_type == 'ANNEXE',
            'page_utile': final_type != 'AUTRE',
            'page_blanche': False
        },
        'image': {
            'largeur_px': page.get('largeur_px'),
            'hauteur_px': page.get('hauteur_px'),
            'rotation_estimee': page.get('rotation_estimee'),
            'deskew_applique': bool(page.get('rotation_estimee'))
        },
        'statut_extraction': {
            'statut': statut,
            'exhaustivite': None,
            'nb_tableaux': nb_tableaux,
            'nb_champs_extraits': nb_champs,
            'commentaire': None
        },
        'entete_page': entete,
        'elements_visuels': visuels,
        'donnees': donnees,
        'tokens_in': rep.get('tokens_in') if rep else 0,
        'tokens_out': rep.get('tokens_out') if rep else 0,
        'temps_s': rep.get('elapsed') if rep else 0.0
    }

def merge_core_tables(base, new):
    if new is None:
        return base
    if base is None:
        return copy.deepcopy(new)
    for key, cols in new.items():
        if key not in base:
            base[key] = copy.deepcopy(cols)
        else:
            for col, val in cols.items():
                if base[key].get(col, {}).get('valeur') is None and val.get('valeur') is not None:
                    base[key][col] = copy.deepcopy(val)
    return base

def build_synthese(page_objects):
    synthese = {
        'actif': None,
        'passif': None,
        'tcr': None,
        'decl': None,
        'annexes': []
    }
    for page in page_objects:
        t = page['classification']['type_page']
        donnees = page.get('donnees', {})
        if t == 'ACTIF' and synthese['actif'] is None:
            synthese['actif'] = donnees.get('actif')
        elif t == 'PASSIF' and synthese['passif'] is None:
            synthese['passif'] = donnees.get('passif')
        elif t == 'TCR':
            synthese['tcr'] = merge_core_tables(synthese['tcr'], donnees.get('tcr'))
        elif t == 'DECL' and synthese['decl'] is None:
            brut = donnees.get('brut')
            if isinstance(brut, dict):
                synthese['decl'] = brut.get('decl')
        elif t == 'ANNEXE':
            synthese['annexes'].append({
                'page_id': page['page_id'],
                'numero_page_scannee': page['numero_page_scannee'],
                'annexe_numero': donnees.get('brut', {}).get('annexe_numero') if isinstance(donnees.get('brut'), dict) else None,
                'titre_page': page['classification'].get('titre_page')
            })
    return synthese

def build_document(pdf_path, pages, page_objects, tok_in, tok_out, elapsed):
    nb_pages = len(pages)
    nb_blanches = sum(1 for p in page_objects if p['classification']['type_page'] == 'BLANCHE')
    nb_utiles = sum(1 for p in page_objects if p['classification']['type_page'] != 'BLANCHE')
    nb_annexes = sum(1 for p in page_objects if p['classification']['type_page'] == 'ANNEXE')
    return {
        'schema_version': '8.0',
        'type_document': 'liasse_fiscale_algerienne_serie_g',
        'document': {
            'document_id': 'doc_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + pdf_path.stem,
            'fichier_source': pdf_path.name,
            'nb_pages_pdf': nb_pages,
            'nb_pages_scannes': nb_pages,
            'nb_pages_blanches': nb_blanches,
            'nb_pages_utiles': nb_utiles,
            'nb_pages_annexes': nb_annexes,
            'date_extraction': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'duree_extraction_s': round(elapsed, 2),
            'modele_extraction': MODEL_PATH.split('/')[-1],
            'prompt_version': 'v8.json_only',
            'referentiel_version': 'liasse_serie_g_v8',
            'tokens_in': tok_in,
            'tokens_out': tok_out,
            'tokens_total': tok_in + tok_out
        },
        'pages': page_objects,
        'synthese': build_synthese(page_objects)
    }

print('✅ Construction pages JSON OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 10 — PIPELINE PRINCIPAL
# Objectif: exécuter le pipeline complet PDF vers JSON.
# Entrées: PDF dans INPUT_DIR.
# Sorties: JSON par dossier + index JSONL.
# Règles: checkpoint JSON, pages blanches non envoyées au VLM.
# ════════════════════════════════════════════════════════════
deja = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter = [p for p in pdfs if p.stem not in deja]
log('A traiter: ' + str(len(a_traiter)) + ' | deja traites: ' + str(len(deja)))

t_total = time.time()
n_ok = n_err = 0
index_records = []

for num, pdf_path in enumerate(a_traiter, start=1):
    t0 = time.time()
    try:
        pages = pdf_to_pages(pdf_path)
        page_objects = []
        active_pages = []

        for p in pages:
            if is_blank(p['image']):
                page_objects.append(build_blank_page(p, pdf_path.name))
            else:
                active_pages.append(p)

        tok_in = 0
        tok_out = 0

        if active_pages:
            mini = [resize(p['image'], 600) for p in active_pages]
            reps1 = []
            for bs in chunks(mini, CLASSIF_BATCH_SIZE):
                reps1 += ask_batch(PROMPT_CLASSIF, bs)
            tok_in += sum(r['tokens_in'] for r in reps1)
            tok_out += sum(r['tokens_out'] for r in reps1)

            pages_typed = []
            for p, rep in zip(active_pages, reps1):
                t = parse_type(rep['text'])
                pages_typed.append((p, t))

            del mini, reps1
            gc.collect()

            groups = {}
            for p, t in pages_typed:
                groups.setdefault(t, []).append(p)

            extracted = {}

            for t, plist in groups.items():
                prompt = EXTRACTION_PROMPTS.get(t, PROMPT_AUTRE)
                for batch in chunks(plist, GPU_BATCH_SIZE):
                    images = [p['image'] for p in batch]
                    reps = ask_batch(prompt, images)
                    for p, rep in zip(batch, reps):
                        tok_in += rep['tokens_in']
                        tok_out += rep['tokens_out']
                        data = parse_json(rep['text'])
                        extracted[p['index']] = build_page_object(p, t, data, rep, pdf_path.name)
                    gc.collect()
                    torch.cuda.empty_cache()

            for p in active_pages:
                if p['index'] in extracted:
                    page_objects.append(extracted[p['index']])
                else:
                    page_objects.append(build_page_object(p, 'AUTRE', {}, None, pdf_path.name))

        page_objects.sort(key=lambda x: x['pdf_page_index'])
        elapsed = time.time() - t0

        result = build_document(
            pdf_path=pdf_path,
            pages=pages,
            page_objects=page_objects,
            tok_in=tok_in,
            tok_out=tok_out,
            elapsed=elapsed
        )

        json_path = JSON_DIR / (pdf_path.stem + '.json')
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2, default=str)

        n_ok += 1

        index_records.append({
            'fichier': pdf_path.name,
            'json_path': str(json_path),
            'nb_pages': len(pages),
            'nb_pages_utiles': result['document']['nb_pages_utiles'],
            'nb_pages_blanches': result['document']['nb_pages_blanches'],
            'nb_pages_annexes': result['document']['nb_pages_annexes'],
            'tokens_total': tok_in + tok_out,
            'duree_s': round(elapsed, 2),
            'date_extraction': result['document']['date_extraction']
        })

        eta = (time.time() - t_total) / num * (len(a_traiter) - num)
        log('[' + str(num).zfill(4) + '/' + str(len(a_traiter)) + '] OK ' + pdf_path.name + ' | ' + str(round(elapsed, 1)) + 's | tok=' + str(tok_in + tok_out) + ' | pages=' + str(len(pages)) + ' | utiles=' + str(result['document']['nb_pages_utiles']) + ' | blanches=' + str(result['document']['nb_pages_blanches']) + ' | annexes=' + str(result['document']['nb_pages_annexes']) + ' | ETA ' + str(round(eta/3600, 2)) + 'h')

    except Exception as e:
        n_err += 1
        log('[' + str(num).zfill(4) + '/' + str(len(a_traiter)) + '] ERREUR ' + pdf_path.name + ' — ' + str(e))
        continue

with open(INDEX_PATH, 'w', encoding='utf-8') as f:
    for rec in index_records:
        f.write(json.dumps(rec, ensure_ascii=False) + chr(10))

log('✅ Termine en ' + str(round(time.time() - t_total, 1)) + 's | OK ' + str(n_ok) + ' | Erreurs ' + str(n_err))
log('Index JSON: ' + str(INDEX_PATH))